# Basic workflow for making a "surrogate" model that learns *both* $d^{4}\sigma^{UU}$ and $\text{BSA}$ as a function of $x_{\text{B}}$, $t$, $Q^{2}$, and $\phi$.

## (1): Import Libraries

In [ ]:
import datetime
import gc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
# it turned out that regularizers hampered the fitting...
from tensorflow.keras import regularizers

from sklearn.model_selection import train_test_split

## (2): Plotting Styles:

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

## (3): Data Loading:

In [ ]:
test_dataframe = pd.read_csv('../data/burner_data.csv')

In [ ]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_beam_unp_target_xsec", "unp_target_bsa"]]

In [ ]:
x_data.head()

In [ ]:
y_data.head()

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data,
    test_size = 0.1, shuffle = True)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = 0.1, shuffle = True)

In [ ]:
len(x_training)

In [ ]:
len(x_validation)

In [ ]:
len(x_testing)

## (4): DNN Stuff:

### (4.1): MSE Loss:

In [ ]:
class MSELoss(tf.keras.losses.Loss):
    def call(self, y_true, y_pred):
        return tf.reduce_mean(tf.square(y_true - y_pred))

### (4.2): DNN Architecture:

In [ ]:
class SurrogateModel(tf.keras.Model):

    # https://keras.io/api/models/model/ -> follow this for custom model architecture

    def __init__(self, cross_section_symmetry_weight = 0.5, bsa_symmetry_weight = 0.5):
        super().__init__()

        self.cross_section_symmetry_weight = cross_section_symmetry_weight
        self.bsa_symmetry_weight = bsa_symmetry_weight

        # regularizer:
        # I learned about regularizers here: https://medium.com/@theo.wolf/physics-informed-neural-networks-a-simple-tutorial-with-pytorch-f28a890b874a
        # weight_regularizer = regularizers.l2(0.01) 

        initializer = tf.keras.initializers.GlorotNormal(seed = None)

        self.dense_layer_1 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_2 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_3 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")

        # linear activation is default activation if `activation` key is not specified: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
        self.cross_section_output = tf.keras.layers.Dense(2, activation = "linear", name = "cross_section")

        # custom loss business:
        self.cross_section_loss_tracker = MSELoss()

    def azimuthal_symmetry_loss(self, X_batch, training = True):
        X_plus = X_batch
        phi = X_batch[:, -1]
        X_minus = tf.concat([X_batch[:, :-1], tf.expand_dims(-phi, axis = 1)], axis = 1)
        y_plus = self(X_plus, training = training)
        y_minus = self(X_minus, training = training)
        return tf.reduce_mean(tf.square(y_plus - y_minus))
    
    def bsa_azimuthal_symmetry_loss(self, X_batch, training = True):
        X_plus = X_batch
        phi = X_batch[:, -1]
        X_minus = tf.concat([X_batch[:, :-1], tf.expand_dims(-phi, axis = 1)], axis = 1)
        y_plus = self(X_plus, training = training)
        y_minus = self(X_minus, training = training)
        return tf.reduce_mean(tf.square(y_plus + y_minus))

    def call(self, inputs, training = False):

        # hidden layer computation:
        hidden_layer = self.dense_layer_1(inputs)
        hidden_layer = self.dense_layer_2(hidden_layer)
        hidden_layer = self.dense_layer_3(hidden_layer)
        cross_section_output = self.cross_section_output(hidden_layer)

        return cross_section_output
    
    def train_step(self, data):

        # unpack data:
        X_batch_data, y_batch_data = data

        with tf.GradientTape() as tape:
            # forward pass:
            predictions = self(X_batch_data, training = True)

            # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
            data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

            # compute phi symmetry loss:
            cross_section_symmetry_loss = self.azimuthal_symmetry_loss(X_batch_data, training = True)

            # compute BSA(phi) symmetry loss:
            bsa_symmetry_loss = self.bsa_azimuthal_symmetry_loss(X_batch_data, training = True)

            # total loss is just a weighted sum:
            total_loss = (
                data_loss + 
                self.cross_section_symmetry_weight * cross_section_symmetry_loss +
                self.bsa_symmetry_weight * bsa_symmetry_loss
                )

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.trainable_variables))

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)

        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "cross_section_symmetry_loss": cross_section_symmetry_loss,
            "bsa_symmetry_loss": bsa_symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

    def test_step(self, data):
        # unpack data:
        X_batch_data, y_batch_data = data
        
        # forward pass evaluation:
        predictions = self(X_batch_data, training = False)

        # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
        data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

         # compute phi symmetry loss:
        cross_section_symmetry_loss = self.azimuthal_symmetry_loss(X_batch_data, training = True)

        # compute BSA(phi) symmetry loss:
        bsa_symmetry_loss = self.bsa_azimuthal_symmetry_loss(X_batch_data, training = True)
        
        # total loss is just a weighted sum:
        total_loss = (
                data_loss + 
                self.cross_section_symmetry_weight * cross_section_symmetry_loss +
                self.bsa_symmetry_weight * bsa_symmetry_loss
                )

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)
            
        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "cross_section_symmetry_loss": cross_section_symmetry_loss,
            "bsa_symmetry_loss": bsa_symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

## (5): **Actually Fitting the Model**:

In [ ]:
NUMBER_OF_EPOCHS = 1250

tf.keras.backend.clear_session()
gc.collect()

dnn_model = SurrogateModel(
    cross_section_symmetry_weight = 0.0,
    bsa_symmetry_weight = 0.0)
dnn_model.compile(
    optimizer = tf.keras.optimizers.Adam(),
    loss = MSELoss())

dnn_model_history = dnn_model.fit(
    x_training, y_training,
    validation_data = (x_validation, y_validation),
    epochs = NUMBER_OF_EPOCHS, batch_size = len(x_training),
    verbose = 1)

number_of_epochs_run = len(dnn_model_history.epoch)
print(f"[INFO]: The model ran for {number_of_epochs_run} epochs before early stopping.")

In [ ]:
model_testing_evaluation_metrics = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
print(f"[INFO]: Evaluation metrics are: {model_testing_evaluation_metrics}")

dictionary_of_keras_metrics = dict(zip(dnn_model.metrics_names, model_testing_evaluation_metrics))
model_testing_loss = dictionary_of_keras_metrics["loss"]

In [ ]:
figure, axis = plt.subplots(1, 1, figsize = (6, 6))

axis.plot(dnn_model_history.history['loss'], 
    label = "Training Loss", color = 'orange', alpha = 0.6)
axis.plot(dnn_model_history.history['val_loss'], 
    label = "Validation Loss", color = 'purple', alpha = 0.6)

axis.set_xlabel("Epoch", fontsize = 14.)
axis.set_ylabel("Loss Values", fontsize = 14.)
axis.set_title(rf"Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
    fontsize = 14.)
axis.legend(fontsize = 14.)
axis.grid(visible = False)

axis.text(
    0.00, -0.11,
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = axis.transAxes,
    verticalalignment = 'top',
    horizontalalignment = 'left',
    fontsize = 9.,
)

for extension in ['png', 'eps']:
    figure.savefig(
        fname = f"./plots/simultaneous_surrogate_v2.{extension}",
        facecolor = 'white',
        transparent = False)

plt.close(figure)

In [ ]:
grouped = test_dataframe.groupby(['t', 'x_b', 'q_squared'])

In [ ]:
phi_smooth = np.linspace(-np.pi, np.pi, 361) 

for (t_val, xb_val, q2_val), group in grouped:
    print(f"Processing t = {t_val}, xb = {xb_val}, Q2 = {q2_val}")

    group = group.sort_values('phi')
    x_group = group[['t', 'x_b', 'q_squared', 'phi']].values
    x_smooth = np.column_stack([
        np.full_like(phi_smooth, t_val),
        np.full_like(phi_smooth, xb_val),
        np.full_like(phi_smooth, q2_val),
        phi_smooth
    ])

    discrete_predcitions = dnn_model.predict(x_group)
    xsec_pred = discrete_predcitions[:, 0]
    bsa_pred  = discrete_predcitions[:, 1]

    smooth_predictions = dnn_model.predict(x_smooth)
    xsec_smooth = smooth_predictions[:, 0]
    bsa_smooth  = smooth_predictions[:, 1]

    phi = group['phi'].values
    xsec_actual = group['unp_beam_unp_target_xsec'].values
    bsa_actual  = group['unp_target_bsa'].values

    xsec_res = xsec_actual - xsec_pred
    bsa_res  = bsa_actual - bsa_pred

    chi2_xsec = np.sum(xsec_res**2) / len(phi)
    chi2_bsa  = np.sum(bsa_res**2) / len(phi)

    residuals_figure, axes = plt.subplots(2, 2, figsize = (10, 8), sharex = 'col')

    axes[0, 0].scatter(phi, xsec_actual, color = 'black', alpha = 0.7, label = 'Data')
    axes[0, 0].plot(phi_smooth, xsec_smooth, color = 'red', lw = 2, label = 'DNN (interpolation)')
    axes[0, 0].set_ylabel(r"$d^{4}\sigma^{UU}$ [nb / GeV$^{4}$]", fontsize = 14)
    axes[0, 0].set_title(rf"Cross Section ($\chi^2_\nu = {chi2_xsec:.3f}$)")
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle = ':', alpha = 0.6)

    axes[0, 1].scatter(phi, xsec_res, color = 'blue', alpha = 0.6)
    axes[0, 1].axhline(0, color = 'black', linestyle = '--')
    axes[0, 1].set_title("Residuals")
    axes[0, 1].grid(True, linestyle = ':', alpha = 0.6)

    axes[1, 0].scatter(phi, bsa_actual, color = 'black', alpha = 0.7, label = 'Data')
    axes[1, 0].plot(phi_smooth, bsa_smooth, color = 'green', lw = 2, label = 'DNN (interpolation)')
    axes[1, 0].set_ylabel("BSA", fontsize = 14)
    axes[1, 0].set_xlabel(r"$\phi$ (radians)", fontsize = 14)
    axes[1, 0].set_title(rf"BSA ($\chi^2_\nu = {chi2_bsa:.3f}$)")
    axes[1, 0].legend()
    axes[1, 0].grid(True, linestyle = ':', alpha = 0.6)

    axes[1, 1].scatter(phi, bsa_res, color = 'purple', alpha = 0.6)
    axes[1, 1].axhline(0, color = 'black', linestyle = '--')
    axes[1, 1].set_xlabel(r"$\phi$ (radians)", fontsize = 14)
    axes[1, 1].set_title("Residuals")
    axes[1, 1].grid(True, linestyle = ':', alpha = 0.6)
    
    residuals_figure.suptitle(
        rf"$t = {t_val:.3f}$, $x_\textrm{{B}} = {xb_val:.3f}$, $Q^2 = {q2_val:.3f}$",
        fontsize = 16
    )

    filename = f"./plots/t{t_val:.3f}_xb{xb_val:.3f}_q2{q2_val:.3f}_residuals_v2"
    for extension in ['png', 'eps']:
        residuals_figure.savefig(
            fname = f"{filename}.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(residuals_figure)